# Extract CSRS-PPP position files

This notebook extracts `.pos` files from the result ZIPs made by `download_csrs_ppp_positions.sh`. It then moves the extracted files into the year/station folders used by the GNSS position notebooks. Change the folder settings below to match where the files live on your computer.

## Extract `.pos` files from the CSRS-PPP ZIPs

In [ ]:
import zipfile
from pathlib import Path

# This folder contains station/year/pos/ result ZIP files.
ROOT = Path("/Volumes/cornice/marquardt/data/GNSS/docker")

if not ROOT.exists():
    raise FileNotFoundError(f"Could not find the CSRS-PPP results folder: {ROOT}")

zip_files = sorted(ROOT.rglob("pos/*.zip"))
print(f"Found {len(zip_files)} ZIP files")

zip_count = 0
files_extracted = 0

for zip_path in zip_files:
    with zipfile.ZipFile(zip_path, "r") as archive:
        pos_members = [name for name in archive.namelist() if name.lower().endswith(".pos")]

        if not pos_members:
            print(f"No .pos file in {zip_path.name}")
            continue

        zip_count += 1
        for member in pos_members:
            output_path = zip_path.parent / Path(member).name

            # Keep an extracted file if this notebook has already run.
            if output_path.exists():
                continue

            with archive.open(member) as source, output_path.open("wb") as destination:
                destination.write(source.read())

            files_extracted += 1
            print(f"Extracted {output_path.name}")

print(f"ZIPs with .pos files: {zip_count}")
print(f".pos files extracted: {files_extracted}")

## Move the `.pos` files into the `refl_code` folder layout

In [ ]:
import shutil
from pathlib import Path

# Source files are station/year/pos/*.pos. The destination is year/pos/station/*.pos.
SRC_ROOT = Path("/Volumes/cornice/marquardt/data/GNSS/docker")
DST_ROOT = Path("/Volumes/cornice/marquardt/data/GNSS/docker/local/refl_code")

if not SRC_ROOT.exists():
    raise FileNotFoundError(f"Could not find the extracted .pos files: {SRC_ROOT}")
if not DST_ROOT.exists():
    raise FileNotFoundError(f"Could not find the refl_code folder: {DST_ROOT}")

pos_files = sorted(SRC_ROOT.glob("*/*/pos/*.pos"))
print(f"Found {len(pos_files)} extracted .pos files")

moved = 0
skipped = 0

for source_path in pos_files:
    station, year, _, filename = source_path.relative_to(SRC_ROOT).parts
    destination_dir = DST_ROOT / year / "pos" / station
    destination_dir.mkdir(parents=True, exist_ok=True)
    destination_path = destination_dir / filename

    # Leave a file in place if an earlier run already moved it.
    if destination_path.exists():
        skipped += 1
        continue

    shutil.move(str(source_path), str(destination_path))
    moved += 1
    print(f"Moved {filename} to {destination_dir}")

print(f".pos files moved: {moved}")
print(f"Existing destination files skipped: {skipped}")